# Libraries

In [1]:
import sys
import os
from IPython.display import clear_output, display
from importnb import Notebook
from pathlib import Path
import re
target_path = r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\LocalizationModel\Functions'
if target_path not in sys.path:
    sys.path.insert(0, target_path)
with Notebook():
    from Functions1 import *
    #print("Success!")

Using device: cuda
Success!


# Datasets Directories

In [2]:
#With Masks
OwnImpainting_PATH = r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\InpaintDataSet'
AinPaint_PATH = r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\AinPaint'
VideoSham_PATH = r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\VideoShamDataset'
FF_PATH= r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\FaceForensics-master'
#REAL
KVid_PATH = r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\Video Processing\DataSets\KoNViD_1k_videos\KoNViD_1k_videos'
ViFCoT_PATH = r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\ViF-CoT-4k\ViF-Bench\source_videos\source_videos\Real'
Panda70_PATH = r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\ViF-CoT-4k\ViF-CoT-4k\source_videos\source_videos\Panda-70M\real'
Kinetics_PATH = r'C:\Users\alex1\OneDrive\Alex\OneDrive\Desktop\ViF-CoT-4k\ViF-CoT-4k\source_videos\source_videos\kinetics\real'

In [3]:
N_FRAMES = 8
RESIZE_SHAPE = (258, 258)  # (height, width)
MASK_THRESHOLD = 0
DIFF_THRESHOLD = 25

# Real and Fake Dictionaries

In [4]:
def natural_key(path):
    return [int(x) if x.isdigit() else x.lower() for x in re.split(r'(\d+)', str(path))]

# Load My Own Impainting Videos

In [5]:
def load_own_inpainting_video(n_frames=N_FRAMES, resize_shape=RESIZE_SHAPE):
    base = Path(OwnImpainting_PATH)
    datasets = [
        (base / "MovingMask" / "results", base / "MovingMask" / "masks"),
        (base / "StaticMask" / "results", base / "StaticMask" / "masks")
    ]
    extensions = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
    candidates = []
    for results_dir, masks_dir in datasets:
        if not results_dir.is_dir() or not masks_dir.is_dir():
            continue
        for result_folder in results_dir.iterdir():
            if not result_folder.is_dir():
                continue
            video_path = result_folder / "inpaint_out.mp4"
            mask_folder_name = result_folder.name[:-4] if result_folder.name.endswith("_cut") else result_folder.name
            mask_dir = masks_dir / mask_folder_name
            if video_path.is_file() and mask_dir.is_dir():
                mask_files = sorted(
                    [p for p in mask_dir.rglob("*") if p.is_file() and p.suffix.lower() in extensions],
                    key=lambda p: str(p).lower()
                )
                if mask_files:
                    candidates.append((video_path, mask_files))
    if not candidates:
        raise FileNotFoundError("No valid Own Inpainting video and corresponding mask folders were found.")
    random.shuffle(candidates)
    for video_path, mask_files in candidates:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            cap.release()
            continue
        frames, masks = [], []
        limit = len(mask_files) if n_frames is None else min(int(n_frames), len(mask_files))
        for i in range(limit):
            ret, frame = cap.read()
            if not ret:
                break
            mask = cv2.imread(str(mask_files[i]), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue
            h, w = resize_shape
            frame = cv2.resize(frame, (w, h), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mask = (mask > MASK_THRESHOLD).astype(np.uint8)
            frames.append(frame)
            masks.append(mask)
        cap.release()
        if frames:
            return frames, masks
    raise RuntimeError("Candidate videos were found, but none could be loaded successfully.")

# AinPaint Dataset

In [6]:
def load_ainpaint_video(n_frames=N_FRAMES, resize_shape=RESIZE_SHAPE):
    frames_root = Path(AinPaint_PATH) / "DiffuEraser" / "432x240"
    masks_root = Path(AinPaint_PATH) / "input_masks" / "resized_432x240"
    if not frames_root.is_dir():
        raise FileNotFoundError(f"AINPAINT frames directory not found: {frames_root}")
    if not masks_root.is_dir():
        raise FileNotFoundError(f"AINPAINT masks directory not found: {masks_root}")
    candidates = []
    for video_dir in frames_root.iterdir():
        if not video_dir.is_dir():
            continue
        mask_dir = masks_root / video_dir.name
        if not mask_dir.is_dir():
            continue
        frame_files = sorted(
            [p for p in video_dir.glob("*.png") if p.is_file()],
            key=natural_key
        )
        mask_files = {
            p.stem: p for p in mask_dir.glob("*.png") if p.is_file()
        }
        pairs = [(p, mask_files[p.stem]) for p in frame_files if p.stem in mask_files]
        if pairs:
            candidates.append((video_dir, pairs))
    random.shuffle(candidates)
    for video_dir, pairs in candidates:
        frames, masks = [], []
        limit = len(pairs) if n_frames is None else min(int(n_frames), len(pairs))
        h, w = resize_shape
        for frame_path, mask_path in pairs[:limit]:
            frame = cv2.imread(str(frame_path), cv2.IMREAD_COLOR)
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if frame is None or mask is None:
                continue
            frame = cv2.resize(frame, (w, h), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            masks.append((mask > MASK_THRESHOLD).astype(np.uint8))
        if frames:
            return frames, masks
    raise RuntimeError(
        "No AINPAINT video folders with matching PNG frame and mask filenames were found."
    )

# VideoSham Dataset

In [7]:
def load_videosham_video(n_frames=N_FRAMES, resize_shape=RESIZE_SHAPE, diff_threshold=DIFF_THRESHOLD):
    base = Path(VideoSham_PATH)
    parts = ["Part1", "Part2"]
    candidates = []
    for part in parts:
        processed_dir = base / f"Processed-{part}"
        original_dir = base / f"Unedited-{part}"
        if not processed_dir.is_dir() or not original_dir.is_dir():
            continue
        originals = {}
        for p in original_dir.glob("*.mp4"):
            match = re.fullmatch(r"(\d+)_unedited", p.stem, flags=re.IGNORECASE)
            if match:
                originals[int(match.group(1))] = p
        for processed_path in processed_dir.glob("*.mp4"):
            match = re.fullmatch(r"(\d+)_processed", processed_path.stem, flags=re.IGNORECASE)
            if not match:
                continue
            video_id = int(match.group(1))
            original_path = originals.get(video_id)
            if original_path is not None:
                candidates.append((processed_path, original_path))
    if not candidates:
        raise FileNotFoundError(
            f"No matching processed/original VideoSham pairs found in {base}"
        )
    random.shuffle(candidates)
    for processed_path, original_path in candidates:
        processed_cap = cv2.VideoCapture(str(processed_path))
        original_cap = cv2.VideoCapture(str(original_path))
        if not processed_cap.isOpened() or not original_cap.isOpened():
            processed_cap.release()
            original_cap.release()
            continue
        processed_count = int(processed_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        original_count = int(original_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        available = min(processed_count, original_count)
        limit = available if n_frames is None else min(int(n_frames), available)
        frames, masks = [], []
        h, w = resize_shape
        for _ in range(limit):
            ret_p, processed_frame = processed_cap.read()
            ret_o, original_frame = original_cap.read()
            if not ret_p or not ret_o:
                break
            if original_frame.shape[:2] != processed_frame.shape[:2]:
                original_frame = cv2.resize(
                    original_frame,
                    (processed_frame.shape[1], processed_frame.shape[0]),
                    interpolation=cv2.INTER_LINEAR
                )
            difference = cv2.absdiff(processed_frame, original_frame)
            difference = np.max(difference, axis=2)
            mask = (difference > diff_threshold).astype(np.uint8)
            processed_frame = cv2.resize(
                processed_frame, (w, h), interpolation=cv2.INTER_LINEAR
            )
            mask = cv2.resize(
                mask, (w, h), interpolation=cv2.INTER_NEAREST
            )
            frames.append(cv2.cvtColor(processed_frame, cv2.COLOR_BGR2RGB))
            masks.append((mask > 0).astype(np.uint8))
        processed_cap.release()
        original_cap.release()
        if frames:
            return frames, masks
    raise RuntimeError("VideoSham pairs were found, but none could be decoded successfully.")

# Face Forensics++ Dataset

In [8]:
def load_ffpp_video(n_frames=N_FRAMES, resize_shape=RESIZE_SHAPE, diff_threshold=DIFF_THRESHOLD):
    base = Path(FF_PATH)
    original_dir = base / "A" / "original_sequences" / "youtube" / "c40" / "videos"
    manipulated_dir = base / "A" / "manipulated_sequences" / "Deepfakes" / "c40" / "videos"
    if not original_dir.is_dir():
        raise FileNotFoundError(f"FF++ original directory not found: {original_dir}")
    if not manipulated_dir.is_dir():
        raise FileNotFoundError(f"FF++ manipulated directory not found: {manipulated_dir}")
    originals = {p.stem: p for p in original_dir.glob("*.mp4") if re.fullmatch(r"\d{3}", p.stem)}
    candidates = []
    for fake_path in manipulated_dir.glob("*.mp4"):
        match = re.fullmatch(r"(\d{3})_(\d{3})", fake_path.stem)
        if match and match.group(1) in originals:
            candidates.append((fake_path, originals[match.group(1)]))
    if not candidates:
        raise FileNotFoundError("No matching FF++ DeepFakes/original video pairs were found.")
    random.shuffle(candidates)
    for fake_path, original_path in candidates:
        fake_cap = cv2.VideoCapture(str(fake_path))
        original_cap = cv2.VideoCapture(str(original_path))
        if not fake_cap.isOpened() or not original_cap.isOpened():
            fake_cap.release()
            original_cap.release()
            continue
        fake_count = int(fake_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        original_count = int(original_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        limit = min(fake_count, original_count)
        if n_frames is not None:
            limit = min(limit, int(n_frames))
        frames, masks = [], []
        h, w = resize_shape
        for _ in range(limit):
            ret_f, fake_frame = fake_cap.read()
            ret_o, original_frame = original_cap.read()
            if not ret_f or not ret_o:
                break
            if original_frame.shape[:2] != fake_frame.shape[:2]:
                original_frame = cv2.resize(
                    original_frame,
                    (fake_frame.shape[1], fake_frame.shape[0]),
                    interpolation=cv2.INTER_LINEAR
                )
            difference = cv2.absdiff(fake_frame, original_frame)
            difference = np.max(difference, axis=2)
            mask = (difference > diff_threshold).astype(np.uint8)
            fake_frame = cv2.resize(fake_frame, (w, h), interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            frames.append(cv2.cvtColor(fake_frame, cv2.COLOR_BGR2RGB))
            masks.append((mask > 0).astype(np.uint8))
        fake_cap.release()
        original_cap.release()
        if frames:
            return frames, masks
    raise RuntimeError("FF++ video pairs were found, but none could be decoded successfully.")

# Artifact Injection

In [9]:
def load_artifact_injection_video(
    n_frames=8, resize_shape=(512, 512), Steps=10,
    Prob=0.5, injections_per_step=1, seed=None
):
    candidates = [
        Path(AinPaint_PATH) / "input_frames" / "432x240_recompressed_hevc",
        Path(VideoSham_PATH) / "Unedited-Part1",
        Path(VideoSham_PATH) / "Unedited-Part2",
        Path(KVid_PATH),
        Path(ViFCoT_PATH),
        Path(FF_PATH) / "A" / "original_sequences" / "youtube" / "c40" / "videos"
    ]
    dataset = random.choice(candidates)
    loaded = get_video_frames(dataset, N=n_frames, size=resize_shape)
    frames = [cv2.cvtColor(Frame2Numpy(v),cv2.COLOR_RGB2BGR) for v in loaded]
    frames, masks = AddInconsistencies_np(
        frames,
        Steps=Steps,
        prob=Prob,
        injections_per_step=injections_per_step,
        seed=seed
    )
    return frames, masks

# Load Random Video

In [10]:
def load_single_video(tf_thresh=0.5, fake_probs=[0.35, 0.45, 0.65, 0.8],
                      n_frames=N_FRAMES, resize_shape=RESIZE_SHAPE,
                      diff_threshold=DIFF_THRESHOLD):
    vid, masks = [], []
    if random.random() > tf_thresh: #Manipulated
        op = random.random()
        if op < fake_probs[0]:
            vid, masks = load_artifact_injection_video(n_frames,resize_shape)
        elif op < fake_probs[1]:
            vid, masks = load_own_inpainting_video(n_frames, resize_shape)
        elif op < fake_probs[2]:
            vid, masks = load_ainpaint_video(n_frames, resize_shape)
        elif op < fake_probs[3]:
            vid, masks = load_videosham_video(n_frames, resize_shape, diff_threshold)
        else:  # FF++ Dataset
            vid, masks = load_ffpp_video(n_frames, resize_shape, diff_threshold)
    else:  # REAL
        dataset = random.choice([
            Path(AinPaint_PATH) / "input_frames" / "432x240_recompressed_hevc",
            Path(VideoSham_PATH) / random.choice(["Unedited-Part1", "Unedited-Part2"]),
            Path(KVid_PATH),
            Path(ViFCoT_PATH),
            Path(Panda70_PATH) / random.choice([
                "CogVideoX1.5-5B-I2V", "CogVideoX1.5-5B-T",
                "HunyuanVideo", "HunyuanVideo-I2V",
                "Wan2.1-T2V-1.3B", "Wan2.2-TI2V-5B-I",
                "Wan2.2-TI2V-5B-T"
            ]),
            Path(Kinetics_PATH) / random.choice([
                "CogVideoX1.5-5B-I2V", "CogVideoX1.5-5B-T",
                "HunyuanVideo", "HunyuanVideo-I2V",
                "Wan2.1-T2V-1.3B", "Wan2.2-TI2V-5B-I",
                "Wan2.2-TI2V-5B-T"
            ]),
            Path(FF_PATH) / "A" / "original_sequences" / "youtube" / "c40" / "videos"
        ])
    
        vid = get_video_frames(dataset, N=n_frames, size=resize_shape)
    
        vid = [
            cv2.cvtColor(
                (Frame2Numpy(v) * 255).clip(0, 255).astype(np.uint8),
                cv2.COLOR_RGB2BGR
            )
            for v in vid
        ]
    
        masks = [
            np.zeros((resize_shape[0], resize_shape[1]), dtype=np.uint8)
            for _ in range(len(vid))
        ]
    return vid, masks

# Feature Extraction

In [11]:
def _resize_map_sequence(data, n_frames, resize_shape, channels=1):
    if isinstance(data, (list, tuple)):
        data = np.stack([np.asarray(x) for x in data], axis=0)
    else:
        data = np.asarray(data)
    if channels == 1:
        if data.ndim == 4 and data.shape[-1] == 1:
            data = data[..., 0]
        elif data.ndim == 4 and data.shape[1] == 1:
            data = data[:, 0]
        if data.ndim == 2:
            data = data[None]
        if data.ndim != 3:
            raise ValueError(f"Unexpected single-channel feature shape: {data.shape}")
    else:
        if data.ndim == 4 and data.shape[1] == channels:
            data = np.moveaxis(data, 1, -1)
        if data.ndim != 4 or data.shape[-1] != channels:
            raise ValueError(f"Unexpected flow shape: {data.shape}")
    if len(data) == n_frames - 1 and channels == 2:
        data = np.concatenate([data, np.zeros_like(data[:1])], axis=0)
    elif len(data) != n_frames:
        raise ValueError(f"Feature temporal mismatch: expected {n_frames} maps, got {len(data)}")
    h, w = resize_shape
    output = []
    for item in data:
        if item.shape[:2] != (h, w):
            item = cv2.resize(item, (w, h), interpolation=cv2.INTER_LINEAR)
        if channels == 1:
            item = item[..., None]
        output.append(item.astype(np.float32))
    return np.stack(output, axis=0)

In [12]:
def _prepare_video(frames, masks, resize_shape):
    if len(frames) != len(masks):
        raise ValueError("Frame and mask counts do not match.")
    h, w = resize_shape
    bgr_frames = []
    rgb_frames = []
    mask_frames = []
    for frame, mask in zip(frames, masks):
        frame = np.asarray(frame)
        mask = np.asarray(mask)
        if frame.ndim != 3 or frame.shape[2] != 3:
            raise ValueError(f"Invalid frame shape: {frame.shape}")
        if frame.shape[:2] != (h, w):
            frame = cv2.resize(frame, (w, h), interpolation=cv2.INTER_LINEAR)
        if mask.ndim == 3:
            mask = mask[..., 0]
        if mask.shape != (h, w):
            mask = cv2.resize(mask.astype(np.uint8),(w, h),interpolation=cv2.INTER_NEAREST)
        frame = np.clip(frame, 0, 255).astype(np.uint8)
        bgr_frames.append(frame)
        rgb_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        mask_frames.append((mask > 0).astype(np.float32))
    return bgr_frames, rgb_frames, np.stack(mask_frames)

In [13]:
def generate_video_batch(batch_size=2,n_frames=N_FRAMES,resize_shape=RESIZE_SHAPE,tf_thresh=0.5,flow_clip=20.0,device="cpu",channels_first=False,
                         return_components=False,max_attempts=None,distribution=[0.35, 0.45, 0.65, 0.8]):
    if batch_size < 1:
        raise ValueError("batch_size must be >= 1")
    if n_frames < 2:
        raise ValueError("n_frames must be >= 2")
    if flow_clip <= 0:
        raise ValueError("flow_clip must be positive")
    if max_attempts is None:
        max_attempts = batch_size * 10
    videos = []
    masks_batch = []
    frames_batch = []
    variance_batch = []
    flow_batch = []
    attempts = 0
    while len(videos) < batch_size and attempts < max_attempts:
        attempts += 1
        try:
            vid, masks = load_single_video(tf_thresh=tf_thresh,n_frames=n_frames,resize_shape=resize_shape,fake_probs=distribution)
            if len(vid) != n_frames or len(masks) != n_frames:
                continue
            vid_bgr, vid_rgb, mask_array = _prepare_video(vid, masks, resize_shape)
            feat_var = feature_3d_spatiotemporal_variance(vid_rgb)
            norm_var = normalize_list(feat_var)
            variance = _resize_map_sequence(norm_var,n_frames=n_frames,resize_shape=resize_shape,channels=1)
            flow = OF_Ferneback_np(frames=vid_bgr)
            flow = _resize_map_sequence(flow,n_frames=n_frames,resize_shape=resize_shape,channels=2)
            flow = np.clip(flow, -flow_clip, flow_clip) / flow_clip
            rgb = np.asarray(vid_rgb, dtype=np.float32) / 255.0
            combined = np.concatenate([rgb, variance, flow], axis=-1 )
            combined = np.moveaxis(combined, -1, 1)
            mask_array = mask_array[:, None]
            videos.append(combined.astype(np.float32))
            masks_batch.append(mask_array.astype(np.float32))
            if return_components:
                frames_batch.append(np.moveaxis(rgb, -1, 1).astype(np.float32))
                variance_batch.append(np.moveaxis(variance, -1, 1).astype(np.float32))
                flow_batch.append(np.moveaxis(flow, -1, 1).astype(np.float32))
        except Exception as e:
            print(f"Video loading attempt {attempts} failed: {e}")
            continue
    if len(videos) != batch_size:
        raise RuntimeError(
            f"Could only load {len(videos)}/{batch_size} videos "
            f"after {attempts} attempts."
        )
    X = torch.from_numpy(np.stack(videos))
    Y = torch.from_numpy(np.stack(masks_batch))
    if channels_first:
        X = X.permute(0, 2, 1, 3, 4).contiguous()
        Y = Y.permute(0, 2, 1, 3, 4).contiguous()
    X = X.to(device)
    Y = Y.to(device)
    if not return_components:
        return X, Y
    components = {
        "frames": torch.from_numpy(np.stack(frames_batch)),
        "variance": torch.from_numpy(np.stack(variance_batch)),
        "flow": torch.from_numpy(np.stack(flow_batch))
    }
    components = {
        key: value.to(device)
        for key, value in components.items()
    }
    if channels_first:
        components = {
            key: value.permute(0, 2, 1, 3, 4).contiguous()
            for key, value in components.items()
        }
    return X, Y, components

# Execution

# Execution

In [17]:
# 8 frames of 256x256

In [18]:
# set de validacion? Maybe/May not be needed
#checar con cada base de datos con experimentos controlados
#checar en paper en que tamano son los videos tamano vs costo
#ir aumentando desde 3 en espacio temporal por cantidad de memoria
#checar que tan dinamica es la escena con una distribucion y outliers
#identificar fondo y primer plano
#checar ultimas publicaciones
#